# A quantum-volume-style test

Quantum volume asks whether a device can run random square circuits (width = depth = m) well enough that more than
two thirds of its output lands in the circuit's heavy outputs, the bitstrings above the median ideal probability. This
notebook builds such circuits from random two-qubit blocks (three CX each, so generic rather than Haar-random) and uses only
a few circuits per width, so it is a relative comparison of the twins' noise, not a certified quantum volume.

In [1]:
from qputwin_demo import *

connect()

QPUTwin at http://localhost:8000: ready, 7 twins reachable


{'status': 'ready',
 'instance': 'c88add07efb8:7:9e791f',
 'job_store': 'ok',
 'backends_reachable': ['quantinuum_helios',
  'ionq_forte',
  'rigetti_ankaa_3',
  'iqm_emerald',
  'ibm_heron',
  'ibm_nighthawk',
  'google_willow']}

## 1. Heavy outputs of one circuit

The ideal probabilities come from the exact statevector; the heavy set is everything above the median. An ideal device lands in it with probability around 0.85. A random circuit can leave a qubit untouched; the engine does not simulate it, and the response's `statevector_qubits` says which qubits the amplitudes cover.

In [2]:
def qv_circuit(m, seed):
    g = np.random.default_rng(seed)
    lines = []
    for _ in range(m):
        perm = g.permutation(m)
        for a, b in zip(perm[0::2], perm[1::2]):
            for _ in range(3):                  # a generic two-qubit block: 1Q layer, CX, three times, then a final 1Q layer
                th = g.uniform(0, 2 * np.pi, 6)
                lines += [f"u3({th[0]},{th[1]},{th[2]}) q[{a}]", f"u3({th[3]},{th[4]},{th[5]}) q[{b}]", f"cx q[{a}],q[{b}]"]
            th = g.uniform(0, 2 * np.pi, 6)
            lines += [f"u3({th[0]},{th[1]},{th[2]}) q[{a}]", f"u3({th[3]},{th[4]},{th[5]}) q[{b}]"]
    return program(m, lines)

def heavy_set(prog, m):
    """Exact output probabilities of all 2^m bitstrings, and the heavy set (strictly above the median)."""
    res = run("quantinuum_helios", prog, shots=1, return_statevector=True)["result"]
    # Only qubits that some gate touches are simulated: `statevector_qubits` lists them (index bit j is
    # statevector_qubits[j]); every untouched qubit is exactly |0>.
    probs = np.zeros(2 ** m)
    for i, (re, im) in enumerate(res["statevector"]):
        full = sum(((i >> j) & 1) << q for j, q in enumerate(res["statevector_qubits"]))
        probs[full] = re * re + im * im
    # the tolerance stops floating-point noise from deciding exact ties at the median
    return {format(i, f"0{m}b") for i in np.flatnonzero(probs > np.median(probs) + 1e-12)}, probs

prog = qv_circuit(4, seed=1)
heavy, probs = heavy_set(prog, 4)
print("heavy outputs:", sorted(heavy))
print("ideal heavy-output probability:", round(sum(probs[int(k, 2)] for k in heavy), 4))

heavy outputs: ['0000', '0011', '0100', '0101', '0111', '1000', '1001', '1010']
ideal heavy-output probability: 0.8369


## 2. Heavy-output probability on every twin

Eight random width-4 circuits per twin under device noise.

In [3]:
def hop(backend, prog, heavy, shots=4000):
    c = run(backend, prog, shots=shots, noise=True, routing=True)["result"]["counts"]
    return sum(v for k, v in c.items() if k in heavy) / sum(c.values())

circuits = [qv_circuit(4, seed=s) for s in range(8)]
heavies = [heavy_set(p, 4)[0] for p in circuits]
rows = []
for b in ALL_BACKENDS:
    h = np.array([hop(b, p, hv) for p, hv in zip(circuits, heavies)])
    rows.append({"twin": NAMES[b], "mean HOP": round(h.mean(), 4), "mean - 2 SE": round(h.mean() - 2 * h.std(ddof=1) / np.sqrt(len(h)), 4)})
t = pd.DataFrame(rows).set_index("twin")
t["passes 2/3"] = t["mean - 2 SE"] > 2 / 3
t

,mean HOP,mean - 2 SE,passes 2/3
twin,,,
Helios,0.8439,0.8029,True
Forte,0.7989,0.7584,True
Ankaa-3,0.7151,0.6709,True
Emerald,0.7596,0.7149,True
Heron r3,0.8293,0.7872,True
Nighthawk,0.8092,0.7675,True
Willow,0.8230,0.7823,True


## 3. Pass or fail by width

Widths 3 to 6 on three twins, twelve circuits each. A width passes when the mean minus two standard errors exceeds 2/3. The first column applies the same test to the exact probabilities, an ideal device: where it fails, the circuits themselves are the reason.

In [4]:
rows = []
for m in (3, 4, 5, 6):
    circuits = [qv_circuit(m, seed=100 * m + s) for s in range(12)]
    sets = [heavy_set(p, m) for p in circuits]
    heavies = [hv for hv, _ in sets]
    ideal = np.array([sum(pr[int(k, 2)] for k in hv) for hv, pr in sets])
    row = {"width": m, "ideal device": "pass" if ideal.mean() - 2 * ideal.std(ddof=1) / np.sqrt(len(ideal)) > 2 / 3 else "fail"}
    for b in ("quantinuum_helios", "ibm_heron", "iqm_emerald"):
        h = np.array([hop(b, p, hv, shots=2000) for p, hv in zip(circuits, heavies)])
        row[NAMES[b]] = "pass" if h.mean() - 2 * h.std(ddof=1) / np.sqrt(len(h)) > 2 / 3 else "fail"
    rows.append(row)
pd.DataFrame(rows).set_index("width")

,ideal device,Helios,Heron r3,Emerald
width,,,,
3,pass,pass,pass,pass
4,pass,pass,pass,pass
5,pass,pass,pass,pass
6,pass,pass,pass,fail


## 4. Why so few circuits is only a comparison

With eight circuits the standard error is large, so a width near the threshold can flip between pass and fail with different random circuits. A certified quantum volume needs at least 100 circuits and Haar-random blocks.

In [5]:
m = 5
for batch in range(3):
    circuits = [qv_circuit(m, seed=1000 + 10 * batch + s) for s in range(8)]
    h = np.array([hop("iqm_emerald", p, heavy_set(p, m)[0], shots=2000) for p in circuits])
    print(f"batch {batch}: mean HOP {h.mean():.4f}, mean - 2 SE {h.mean() - 2 * h.std(ddof=1) / np.sqrt(8):.4f}")

batch 0: mean HOP 0.7383, mean - 2 SE 0.7004


batch 1: mean HOP 0.6917, mean - 2 SE 0.6778


batch 2: mean HOP 0.7373, mean - 2 SE 0.7133


## Takeaways

- Heavy-output tests compare the twins' noise on random circuits.
- Few circuits and generic blocks make this a relative test, not a certified quantum volume.